# Code to modify one of the yaml files so that it has the correct formatting for the current version of cantera

In [1]:
from ruamel.yaml import YAML
from ruamel.yaml.comments import CommentedMap
import numpy as np

import sys

file_path = '/home/austin/Desktop/Kreitz Lab Work/methanation-model/data/File_0_chem_covdep_unmodified.yaml'
file_out = '/home/austin/Desktop/Kreitz Lab Work/methanation-model/data/File_0_chem_covdep_edited_automatically.yaml'

Test out the manually edited yaml file

In [2]:
yaml = YAML()

# Reading data from the YAML file
with open('/home/austin/Desktop/Kreitz Lab Work/temp-methanation-model/data/File_0_chem_covdep_edited_manually.yaml', 'r') as file:
    data = yaml.load(file)

Read in the yaml file

In [3]:
yaml = YAML()

# Reading data from the YAML file
with open(file_path, 'r') as file:
    data = yaml.load(file)

edit the yaml file to match the modern cantera formatting

In [4]:
# add the following to the surface phase :   adjacent-phases: [gas]
for phase in data['phases']:
    if phase['name'] == 'surface1':
        phase['adjacent-phases'] = ['gas']

#change the coverage dependency formatting
for specie in data['species']:
    if specie['name'] == 'OX(10)':

        ox = specie.get('coverage-dependencies')
        dep = ox.get('OX(10)')
        h1 = dep.get('enthalpy-1st-order')
        h2 = dep.get('enthalpy-2nd-order')
        
        new_dep = CommentedMap()        
        new_dep["model"] = 'polynomial'
        new_dep["units"] = CommentedMap([("energy", 'eV'), ("quantity", "molec")])
        new_dep["enthalpy-coefficients"] = [h1, h2, 0, 0.0]

        new_dep.fa.set_flow_style()
        new_dep["units"].fa.set_flow_style()

        ox[specie['name']] = new_dep

    if specie['name'] == 'OCX(11)':
        
        ocx = specie.get('coverage-dependencies')
        dep = ocx.get('OCX(11)')
        h1 = dep.get('enthalpy-1st-order')
        h2 = dep.get('enthalpy-2nd-order')
        
        new_dep = CommentedMap()        
        new_dep["model"] = 'polynomial'
        new_dep["units"] = CommentedMap([("energy", 'eV'), ("quantity", "molec")])
        new_dep["enthalpy-coefficients"] = [h1, h2, 0, 0.0]

        new_dep.fa.set_flow_style()
        new_dep["units"].fa.set_flow_style()

        ocx[specie['name']] = new_dep

Save the file

In [5]:
with open(file_out,'w') as f_out:
    yaml.dump(data,f_out)

Verify that the yaml file created by this code results in the same cantera interface object

In [6]:
import cantera as ct

surf1 = ct.Interface('/home/austin/Desktop/Kreitz Lab Work/temp-methanation-model/data/File_0_chem_covdep_edited_manually.yaml', 'surface1')
surf2 = ct.Interface(file_out,'surface1')

#check that the two files result in the same surface object. Since there is no direct way to check equality we just check a couple of different things

print(surf1.T == surf2.T)
print(surf1.P == surf2.P)
print(all(surf1.X == surf2.X))
print(surf1.h == surf2.h)
print(np.all(np.array(surf1.species_names) == np.array(surf2.species_names)))
print(surf1.reaction_equations() == surf2.reaction_equations())

True
True
True
True
True
True


Additionally, double check that the coverage dependence is working

In [7]:
species_name = 'OX(10)'
i_sp = surf2.species_index(species_name)

print('no coverage')
h_rt = surf2.standard_enthalpies_RT[i_sp]
print(h_rt)

print('Half Coverage')
surf2.coverages = {species_name: 1, 'CX(14)':1}
h_rt = surf2.standard_enthalpies_RT[i_sp]
print(h_rt)

print('full coverage')
surf2.coverages = {species_name: 1}
h_rt = surf2.standard_enthalpies_RT[i_sp]
print(h_rt)

no coverage
-94.884745307662
Half Coverage
-78.86534619265448
full coverage
-38.01491232973474


Check the elements section of the yaml file

In [8]:
index = surf1.element_index('X')
gas = surf1.adjacent['gas']
print(gas.atomic_weight(index))

195.083


In [9]:
r = ct.IdealGasReactor(gas, energy='off', clone=True)


CanteraError: 
*******************************************************************************
CanteraError thrown by getElementWeight:
element not found: X
*******************************************************************************
